# **Introdução ao LangChain**

**Disciplina:** Generative AI & Advanced Analytics

**Instituição:** PUC Minas

**Professor:** Renan Santos Mendes

**Email:** renansantosmendes@gmail.com

---

Este notebook tem como objetivo apresentar de forma pratica os conceitos iniciais do LangChain, uma biblioteca para construcao de aplicacoes baseadas em modelos de linguagem (LLMs).

Serao abordados os seguintes topicos:

1. Configuracao do ambiente e do modelo de linguagem
2. Messages (SystemMessage, HumanMessage e AIMessage)
3. Templates (PromptTemplate e ChatPromptTemplate)
4. Runnables (conceitos basicos)
5. Chains (encadeamento de componentes)


## 1. Configuracao do ambiente

Antes de comecar, precisamos instalar a biblioteca `langchain-openai`, que fornece a integracao entre o LangChain e modelos compativeis com a API da OpenAI.

Nesta aula, usaremos um proxy proprio para acessar o modelo `gpt-4o-mini`, entao nao sera necessario fornecer uma chave de API valida.

A instalacao sera feita utilizando o `uv`, um gerenciador de pacotes mais rapido que o `pip` tradicional.

In [1]:
!uv pip install langchain-openai -q

'uv' is not recognized as an internal or external command,
operable program or batch file.


## 2. Criando o modelo de linguagem

O `ChatOpenAI` e a classe do LangChain responsavel por representar um modelo de chat compativel com a API da OpenAI.

Abaixo, criamos uma instancia do modelo apontando para o proxy da disciplina.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key="not-needed",
    base_url="https://pgl-proxy.vercel.app/v1",
)

Podemos testar o modelo enviando uma pergunta simples diretamente como texto.

In [ ]:
response = ...
print(response.content)

## 3. Messages

Ao trabalhar com modelos de chat, a comunicacao e organizada em **mensagens**, cada uma com um papel especifico dentro da conversa. As tres principais mensagens do LangChain sao:

- **SystemMessage**: define o comportamento, o tom ou as regras que o modelo deve seguir durante toda a conversa. E como se fosse uma instrucao dada ao modelo antes do dialogo comecar.
- **HumanMessage**: representa a fala do usuario, ou seja, a pergunta ou solicitacao feita ao modelo.
- **AIMessage**: representa a resposta gerada pelo modelo. Tambem pode ser usada para simular respostas anteriores do modelo em um historico de conversa.

Vamos importar essas classes e construir uma conversa simples.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

messages = ...

response = llm.invoke(messages)
print(response.content)

Podemos tambem simular um historico de conversa, incluindo uma resposta anterior do modelo (`AIMessage`) antes de fazer uma nova pergunta. Isso ajuda o modelo a manter contexto sobre o que ja foi dito.

In [ ]:
conversation_history = ...

response = llm.invoke(conversation_history)
print(response.content)

## 4. Templates

Em aplicacoes reais, raramente escrevemos prompts fixos: normalmente queremos reaproveitar uma mesma estrutura de prompt, alterando apenas alguns valores. Para isso, o LangChain oferece os **templates**.

- **PromptTemplate**: usado para criar um template de texto simples, com variaveis que serao preenchidas dinamicamente.
- **ChatPromptTemplate**: usado para criar um template composto por varias mensagens (system, human, etc.), tambem com variaveis dinamicas.

Vamos comecar com o `PromptTemplate`.

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = ...

formatted_prompt = prompt_template.format(concept="loop")
print(formatted_prompt)

Agora vamos usar o `ChatPromptTemplate`, que permite definir varias mensagens dentro do mesmo template, cada uma com seu papel (system, human, etc.).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt_template = ...

formatted_messages = chat_prompt_template.format_messages(concept="funcao")
for message in formatted_messages:
    print(message)

## 5. Runnables

Todo componente do LangChain que pode ser executado (um modelo, um template, uma funcao de transformacao de dados, entre outros) implementa a interface `Runnable`.

Um `Runnable` expoe metodos padronizados, como:

- `invoke`: executa o componente com uma unica entrada
- `batch`: executa o componente com varias entradas de uma vez
- `stream`: executa o componente retornando a resposta em partes (streaming)

Essa padronizacao e o que permite combinar diferentes componentes entre si, formando as **chains**, que veremos a seguir.

Abaixo, um exemplo simples mostrando que tanto o `ChatPromptTemplate` quanto o `llm` sao `Runnables`, pois ambos possuem o metodo `invoke`.

In [ ]:
print(hasattr(chat_prompt_template, "invoke"))
print(hasattr(llm, "invoke"))

## 6. Chains

Uma **chain** e o encadeamento de dois ou mais `Runnables`, de forma que a saida de um componente seja usada como entrada do proximo.

No LangChain, esse encadeamento e feito de maneira declarativa usando o operador `|` (pipe), conhecido como LCEL (LangChain Expression Language).

Abaixo, vamos criar uma chain simples que:

1. Recebe um conceito
2. Formata o prompt usando o `ChatPromptTemplate`
3. Envia o prompt formatado para o modelo `llm`

In [ ]:
chain = ...

response = chain.invoke({"concept": "recursao"})
print(response.content)

Podemos tambem adicionar mais um passo a chain, por exemplo, extraindo apenas o texto da resposta usando o `StrOutputParser`, que converte a saida do modelo (um `AIMessage`) em uma string simples.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain_with_parser = ...

result = chain_with_parser.invoke({"concept": "lista encadeada"})
print(result)
print(type(result))

## 7. Runnables avançados: RunnableParallel e RunnableBranch

Alem do encadeamento simples com o operador `|`, o LangChain oferece componentes que permitem organizar a execucao dos `Runnables` de formas mais elaboradas. Dois exemplos bastante uteis sao:

- **RunnableParallel**: executa varios `Runnables` ao mesmo tempo, usando a mesma entrada, e retorna um dicionario com o resultado de cada um. E util quando queremos, por exemplo, gerar respostas diferentes para o mesmo conceito (uma explicacao e um exemplo de codigo, ao mesmo tempo).
- **RunnableBranch**: permite definir diferentes caminhos de execucao (chains diferentes) de acordo com uma condicao aplicada sobre a entrada. Funciona como uma estrutura de `if / elif / else` para `Runnables`.

Vamos ver um exemplo simples de cada um.

### 7.1 RunnableParallel

No exemplo abaixo, vamos executar duas chains diferentes ao mesmo tempo, a partir do mesmo conceito de entrada:

- uma chain que gera uma explicacao teorica
- uma chain que gera um exemplo de codigo em Python

O resultado sera um dicionario contendo as duas respostas.

In [ ]:
from langchain_core.runnables import RunnableParallel

explanation_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um professor de programacao."),
    ("human", "Explique de forma breve o conceito de {concept}."),
])

code_example_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um professor de programacao."),
    ("human", "Escreva um exemplo curto de codigo em Python sobre {concept}."),
])

explanation_chain = explanation_prompt | llm | StrOutputParser()
code_example_chain = code_example_prompt | llm | StrOutputParser()

parallel_chain = ...

parallel_result = parallel_chain.invoke({"concept": "list comprehension"})
print(parallel_result["explanation"])
print("---")
print(parallel_result["code_example"])

### 7.2 RunnableBranch

No exemplo abaixo, vamos criar duas chains diferentes: uma especializada em conceitos de programacao e outra especializada em conceitos de matematica. O `RunnableBranch` sera responsavel por escolher qual chain executar, de acordo com o valor do campo `topic` presente na entrada.

Cada condicao do `RunnableBranch` e uma tupla no formato `(funcao_condicao, chain)`. A ultima entrada, sem condicao, funciona como o caminho padrao (`else`).

In [ ]:
from langchain_core.runnables import RunnableBranch

programming_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um professor de programacao."),
    ("human", "Explique de forma breve o conceito de {concept}."),
])

math_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um professor de matematica."),
    ("human", "Explique de forma breve o conceito de {concept}."),
])

default_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um assistente generalista."),
    ("human", "Explique de forma breve o conceito de {concept}."),
])

programming_chain = programming_prompt | llm | StrOutputParser()
math_chain = math_prompt | llm | StrOutputParser()
default_chain = default_prompt | llm | StrOutputParser()

branch_chain =...

programming_result = branch_chain.invoke({"topic": "programming", "concept": "recursao"})
print(programming_result)
print("---")
math_result = branch_chain.invoke({"topic": "math", "concept": "derivada"})
print(math_result)